In [3]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import copy
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) # normalização do imagenet
])

train_path = './cifake/train'
train_dataset = ImageFolder(root=train_path, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_path = './cifake/test'
val_dataset = ImageFolder(root=val_path, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

class EfficientNetV2Detector(nn.Module):
    def __init__(self,):
        super(EfficientNetV2Detector, self).__init__()
        
        self.base_model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        
        num_features = self.base_model.classifier[1].in_features # número de features da camada final do efficientnet
        self.base_model.classifier = nn.Identity() # substitui a camada final (fc) por uma identidade para extrair as features
        
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),    
            nn.Linear(256, 64),          
            nn.ReLU(),
            nn.Linear(64, 1)             
        )

    def forward(self, x):
        x = self.base_model(x)
        x = self.classifier(x)
        return x

model = EfficientNetV2Detector().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adamax(model.parameters(), lr=0.001)

def train_model(model, optimizer, max_epochs=20, patience=3):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        print(f'\nEpoch {epoch+1}/{max_epochs}')

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            correct = 0 
            total = 0

            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.unsqueeze(1).float().to(device)

                optimizer.zero_grad()

                # calcula o gradiente só se tiver em treino
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item()
                
                with torch.no_grad():
                    predicted = torch.round(torch.sigmoid(outputs.data)) 
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            epoch_loss = running_loss / len(dataloader)
            epoch_acc = correct / total

            print(f'{phase.capitalize()} -> Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')

            if phase == 'val':
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict()) # copia o melhor modelo
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    print(f'Sem melhoria por {epochs_no_improve} época(s).')

        if epochs_no_improve >= patience:
            print('\nEarly Stopping ativado! Interrompendo o treinamento.')
            break

    model.load_state_dict(best_model_wts)
    return model

print("\nTreinando classifier")
for param in model.base_model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer_fc = optim.Adamax(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

model = train_model(model, optimizer_fc, max_epochs=20, patience=1)

def metrics(model, dataloader, device, dataset_name):
    model.eval() 
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            
            outputs = model(inputs)
        
            preds = torch.round(torch.sigmoid(outputs))
            
            all_preds.extend(preds.squeeze().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print("-------------------------------------")
    print(f"Relatório Final: {dataset_name}")
    
    report = classification_report(all_labels, all_preds, target_names=['Fake', 'Real'], digits=4)
    print(report)

metrics(model, val_loader, device, "Teste")

metrics(model, train_loader, device, "Treino")

print("\n==================Fine tuning==================")
for param in model.base_model.parameters():
    param.requires_grad = True

optimizer_full = optim.Adamax(model.parameters(), lr=0.001)

model = train_model(model, optimizer_full, max_epochs=20, patience=1)

metrics(model, val_loader, device, "Teste")

metrics(model, train_loader, device, "Treino")



Treinando classifier

Epoch 1/20
Train -> Loss: 0.6437, Accuracy: 0.6112
Val -> Loss: 0.6121, Accuracy: 0.6866

Epoch 2/20
Train -> Loss: 0.6308, Accuracy: 0.6254
Val -> Loss: 0.6020, Accuracy: 0.6790

Epoch 3/20
Train -> Loss: 0.6283, Accuracy: 0.6289
Val -> Loss: 0.8306, Accuracy: 0.6773
Sem melhoria por 1 época(s).

Early Stopping ativado! Interrompendo o treinamento.
-------------------------------------
Relatório Final: Teste
              precision    recall  f1-score   support

        Fake     0.7283    0.5708    0.6400     10000
        Real     0.6471    0.7871    0.7103     10000

    accuracy                         0.6790     20000
   macro avg     0.6877    0.6789    0.6752     20000
weighted avg     0.6877    0.6790    0.6752     20000

-------------------------------------
Relatório Final: Treino
              precision    recall  f1-score   support

        Fake     0.7244    0.5755    0.6414     50000
        Real     0.6478    0.7810    0.7082     50000

    accurac